## Validación del aprendizaje federado con múltiples rondas

El objetivo es analizar el efecto acumulado de la agregación iterativa mediante FedAvg sobre el rendimiento del modelo y sobre las métricas de sostenibilidad. En lugar de realizar una única ronda de entrenamiento local seguida de una agregación, el proceso federado se repite durante varias rondas, permitiendo observar la evolución del modelo global a lo largo del tiempo.

La configuración experimental utilizada es la siguiente:

- 3 clientes federados.
- 3 rondas de comunicación.
- 3 épocas de entrenamiento local por cliente y por ronda.
- Agregación FedAvg al final de cada ronda.

CodeCarbon permanece integrado en cada cliente para recopilar métricas de sostenibilidad durante todo el proceso de entrenamiento. De esta forma, es posible obtener medidas tanto a nivel de cliente como del sistema completo, incluyendo consumo energético total, emisiones operativas y tiempo de ejecución acumulado en todas las rondas federadas.

Este notebook representa la primera implementación completa de un flujo de aprendizaje federado iterativo, proporcionando una aproximación más realista al comportamiento de los sistemas de aprendizaje federado y permitiendo estudiar la relación entre el número de rondas de comunicación, la precisión del modelo y las métricas de sostenibilidad.

0. Constantes de definición del entorno

In [1]:
DATA_DIR = "../data"
PROBES_DIR = "../probes"

DATASET = "CIFAR10"
SCENARIO = "FL_7"
MODEL_NAME = "SimpleCNN"
NUM_CLIENTS = 7

BATCH_SIZE = 64
EPOCHS = 2
NUM_ROUNDS = 2
LEARNING_RATE = 0.001

RANDOM_SEED = 42

## Temporal

DEVICE_CATEGORY = "laptop"
DEVICE_YEAR = 2023

1. Carga del Dataset y DataLoader y partición del modelo

In [2]:
import torch

from torch.utils.data import DataLoader
from torch.utils.data import random_split

from torchvision import datasets
from torchvision import transforms


# ==================================================
# Transformaciones
# ==================================================

transform = transforms.ToTensor()

# ==================================================
# Dataset
# ==================================================

train_dataset = datasets.CIFAR10(
    root=DATA_DIR,
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.CIFAR10(
    root=DATA_DIR,
    train=False,
    download=True,
    transform=transform
)

# ==================================================
# Partición FL-#
# ==================================================

client_size = len(train_dataset) // NUM_CLIENTS

lengths = [client_size] * NUM_CLIENTS

lengths[-1] += (
    len(train_dataset) - sum(lengths)
)

client_datasets = random_split(
    train_dataset,
    lengths
)


# ==================================================
# DataLoaders de clientes
# ==================================================

client_loaders = [
    DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=True
    )
    for ds in client_datasets
]

# ==================================================
# Test loader global
# ==================================================

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


2. Modelo

In [3]:
import torch.nn as nn

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

3. Modelo local para pruebas

In [4]:
model = SimpleCNN()

print(model)

SimpleCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=4096, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=10, bias=True)
  )
)


4. Función de pérdida y optimización

In [5]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

5. Función de entrenamiento

In [6]:
def train_model(
    model,
    train_loader,
    optimizer,
    criterion,
    epochs
):

    model.train()

    for epoch in range(epochs):

        running_loss = 0.0

        for images, labels in train_loader:

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

        epoch_loss = running_loss / len(train_loader)

        print(
            f"Epoch {epoch + 1}/{epochs} - "
            f"Loss: {epoch_loss:.4f}"
        )

6. Función de evaluación

In [7]:
def evaluate_model(
    model,
    test_loader
):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in test_loader:

            outputs = model(images)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)

            correct += (
                predicted == labels
            ).sum().item()

    accuracy = correct / total

    return accuracy

7. Función de entrenamiento y captura de datos de emisiones

In [ ]:
from codecarbon import EmissionsTracker
from pathlib import Path

def run_training_with_codecarbon(
    train_function,
    probe_id,
    scenario,
    client_id=1,
    output_dir=PROBES_DIR
):
    Path(output_dir).mkdir(
    parents=True,
    exist_ok=True
    )
    
    tracker = EmissionsTracker(
        output_dir=output_dir,
        output_file=(
            f"cc_{probe_id}_{scenario}_{client_id}.csv"
        )
    )

    tracker.start()

    tracker.start_task("training")

    train_function()

    training_data = tracker.stop_task()

    tracker.stop()

# Eliminar archivos de emisiones base     
    for file in Path(output_dir).glob("emissions_base*.csv"):
        file.unlink()

    return {
        "client_id": client_id,
        "energy_total_kwh": training_data.energy_consumed,
        "emissions_total_kg": training_data.emissions,
        "training_time_total_s": training_data.duration
}


8. Cliente Flower

In [9]:
import flwr as fl

class FlowerClient(fl.client.NumPyClient):

    def __init__(
        self,
        model,
        train_loader,
        test_loader,
        probe_id,
        client_id
    ):
        self.model = model
        self.train_loader = train_loader
        self.test_loader = test_loader
        self.probe_id = probe_id
        self.client_id = client_id

    def get_parameters(
        self,
        config
    ):
        return [
            val.detach().numpy()
            for _, val in self.model.state_dict().items()
        ]

    def set_parameters(
        self,
        parameters
    ):

        params_dict = zip(
            self.model.state_dict().keys(),
            parameters
        )

        state_dict = {
            k: torch.tensor(v)
            for k, v in params_dict
        }

        self.model.load_state_dict(
            state_dict,
            strict=True
        )

    def fit(
        self,
        parameters,
        config
    ):

        self.set_parameters(parameters)

        local_criterion = nn.CrossEntropyLoss()

        local_optimizer = optim.Adam(
            self.model.parameters(),
            lr=LEARNING_RATE
        )

        cc_metrics = run_training_with_codecarbon(
            train_function=lambda: train_model(
                self.model,
                self.train_loader,
                local_optimizer,
                local_criterion,
                EPOCHS
            ),
            probe_id=self.probe_id,
            scenario=SCENARIO,
            client_id=self.client_id,
            output_dir=PROBES_DIR
        )

        return (
            self.get_parameters(config),
            len(self.train_loader.dataset),
            cc_metrics
        )

    def evaluate(
        self,
        parameters,
        config
    ):

        self.set_parameters(parameters)

        accuracy = evaluate_model(
            self.model,
            self.test_loader
        )

        return (
            0.0,
            len(self.test_loader.dataset),
            {
                "accuracy": accuracy
            }
        )

7. Función FedAvg

In [10]:
import numpy as np

def fedavg(client_parameters):

    aggregated_parameters = []

    for layer_weights in zip(*client_parameters):

        aggregated_layer = np.mean(
            layer_weights,
            axis=0
        )

        aggregated_parameters.append(
            aggregated_layer
        )

    return aggregated_parameters

8. Inicialización del Entorno Federado

In [11]:
def initialize_federated_environment(
        probe_id
):

    # Modelo global inicial
    global_model = SimpleCNN()

    global_parameters = [
        val.detach().numpy()
        for _, val in global_model.state_dict().items()
    ]

    # Clientes
    clients = []

    for i in range(NUM_CLIENTS):

        clients.append(
            FlowerClient(
                model=SimpleCNN(),
                train_loader=client_loaders[i],
                test_loader=test_loader,
                probe_id=probe_id,
                client_id=i + 1
            )
        )

    return (
        global_model,
        global_parameters,
        clients
    )

9. Ejecución del entrenamiento generado

In [12]:
def run_federated_training(
    clients,
    global_parameters
):

    client_metrics = []

    for round_num in range(NUM_ROUNDS):

        round_parameters = []

        round_metrics = []

        print(
            f"\n===== ROUND {round_num + 1} ====="
        )

        for client_id, client in enumerate(clients):

            updated_parameters, num_examples, metrics = client.fit(
                global_parameters,
                {}
            )

            metrics["round"] = round_num + 1
            metrics["client_id"] = client_id + 1

            round_metrics.append(
                metrics
            )

            round_parameters.append(
                updated_parameters
            )

        global_parameters = fedavg(
            round_parameters
        )

        client_metrics.extend(
            round_metrics
        )

    return (
        global_parameters,
        client_metrics
    )

10. Construcción del modelo global

In [13]:
def build_global_model(
    global_parameters
):

    global_model = SimpleCNN()

    params_dict = zip(
        global_model.state_dict().keys(),
        global_parameters
    )

    state_dict = {
        k: torch.tensor(v)
        for k, v in params_dict
    }

    global_model.load_state_dict(
        state_dict,
        strict=True
    )

    return global_model

11. Ejecución del experimento federado

In [14]:
import uuid

# ==================================================
# Identificador único del experimento
# ==================================================

probe_id = str(uuid.uuid4())

# ==================================================
# Inicialización
# ==================================================

global_model, global_parameters, clients = (
    initialize_federated_environment(
        probe_id
    )
)

# ==================================================
# Entrenamiento federado
# ==================================================

global_parameters, client_metrics = (
    run_federated_training(
        clients,
        global_parameters
    )
)

# ==================================================
# Construcción del modelo global
# ==================================================

global_model = build_global_model(
    global_parameters
)

# ==================================================
# Evaluación final
# ==================================================

global_accuracy = evaluate_model(
    global_model,
    test_loader
)

print(
    f"Global Accuracy: {global_accuracy:.4f}"
)

[codecarbon WARNING @ 13:07:44] Multiple instances of codecarbon are allowed to run at the same time.



===== ROUND 1 =====


[codecarbon INFO @ 13:07:45] Emissions data (if any) will be saved to file /home/rutpal/TFM/probes/cc_ff152fbb-0a31-4ce7-a190-19f3aadc9ca5_FL_7_1.csv
[codecarbon INFO @ 13:07:45] [setup] RAM Tracking...
[codecarbon INFO @ 13:07:45] [setup] CPU Tracking...
[codecarbon INFO @ 13:07:45] [setup] GPU Tracking...
[codecarbon INFO @ 13:07:45] No GPU found.
[codecarbon WARNING @ 13:07:46] We saw that you have a 12th Gen Intel(R) Core(TM) i7-1255U but we don't know it. Please contact us.
[codecarbon WARNING @ 13:07:46] We will use the default power consumption of 4 W per thread for your 12 CPU, so 48W.
[codecarbon WARNING @ 13:07:46] No CPU tracking mode found. Falling back on estimation based on TDP for CPU. 
 Linux OS detected: Please ensure RAPL files exist, and are readable, at /sys/class/powercap/intel-rapl/subsystem to measure CPU

[codecarbon INFO @ 13:07:46] CPU Model on constant consumption mode: 12th Gen Intel(R) Core(TM) i7-1255U
[codecarbon WARNING @ 13:07:46] No CPU tracking mode f

Epoch 1/2 - Loss: 1.9404


[codecarbon INFO @ 13:07:59] Energy consumed for RAM : 0.000029 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 13:07:59] Delta energy consumed for CPU with cpu_load : 0.000023 kWh, power : 7.957350624000001 W
[codecarbon INFO @ 13:07:59] Energy consumed for All CPU : 0.000023 kWh
[codecarbon INFO @ 13:07:59] 0.000052 kWh of electricity and 0.000000 L of water were used since the beginning.
[codecarbon WARNING @ 13:07:59] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 13:07:59] Emissions data (if any) will be saved to file /home/rutpal/TFM/probes/cc_ff152fbb-0a31-4ce7-a190-19f3aadc9ca5_FL_7_2.csv
[codecarbon INFO @ 13:07:59] >>> Tracker's metadata:
[codecarbon INFO @ 13:07:59]   Platform system: Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39
[codecarbon INFO @ 13:07:59]   Python version: 3.12.3
[codecarbon INFO @ 13:07:59]   CodeCarbon version: 3.3.1
[codecarbon INFO @ 13:07:59]   Available RAM : 7.632 GB
[codecarbon INFO @ 13:07:59]  

...

Epoch 2/2 - Loss: 1.4121
Epoch 1/2 - Loss: 1.5888


[codecarbon INFO @ 13:10:07] Energy consumed for RAM : 0.000027 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 13:10:07] Delta energy consumed for CPU with cpu_load : 0.000021 kWh, power : 7.8239887248 W
[codecarbon INFO @ 13:10:07] Energy consumed for All CPU : 0.000021 kWh
[codecarbon INFO @ 13:10:07] 0.000049 kWh of electricity and 0.000000 L of water were used since the beginning.
[codecarbon WARNING @ 13:10:07] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 13:10:07] Emissions data (if any) will be saved to file /home/rutpal/TFM/probes/cc_ff152fbb-0a31-4ce7-a190-19f3aadc9ca5_FL_7_7.csv
[codecarbon INFO @ 13:10:07] >>> Tracker's metadata:
[codecarbon INFO @ 13:10:07]   Platform system: Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39
[codecarbon INFO @ 13:10:07]   Python version: 3.12.3
[codecarbon INFO @ 13:10:07]   CodeCarbon version: 3.3.1
[codecarbon INFO @ 13:10:07]   Available RAM : 7.632 GB
[codecarbon INFO @ 13:10:07]   CPU 

Epoch 2/2 - Loss: 1.4117
Epoch 1/2 - Loss: 1.5802


[codecarbon INFO @ 13:10:19] Energy consumed for RAM : 0.000027 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 13:10:19] Delta energy consumed for CPU with cpu_load : 0.000022 kWh, power : 7.85764564608 W
[codecarbon INFO @ 13:10:19] Energy consumed for All CPU : 0.000022 kWh
[codecarbon INFO @ 13:10:19] 0.000049 kWh of electricity and 0.000000 L of water were used since the beginning.


Epoch 2/2 - Loss: 1.4032
Global Accuracy: 0.5312


12. Captura de datos del probe (code carbon (cc) y entorno (env))

In [15]:
import time
from pathlib import Path
import pandas as pd
import uuid

#global_accuracy = evaluate_model(
#    global_model,
#    test_loader
#)

energy_total_kwh = sum(
    m["energy_total_kwh"]
    for m in client_metrics
)

emissions_total_kg = sum(
    m["emissions_total_kg"]
    for m in client_metrics
)

training_time_total_s = sum(
    m["training_time_total_s"]
    for m in client_metrics
)

env = {

    # Identificación
    "probe_id": probe_id,
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),

    # Dispositivo
    "device_category": DEVICE_CATEGORY,
    "device_year": DEVICE_YEAR,

    # Escenario experimental
    "scenario": SCENARIO,
    "dataset": DATASET,
    "model": MODEL_NAME,
    
    # Topología FL

    "num_clients": NUM_CLIENTS,
    "num_rounds": NUM_ROUNDS,


    # Configuración de entrenamiento
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,

    # Tamaño del trabajo
    "training_samples": len(train_dataset),

    # Resultados
    "accuracy": global_accuracy,
    "energy_total_kwh": energy_total_kwh,
    "emissions_total_kg": emissions_total_kg,
    "training_time_total_s": training_time_total_s
}

probes_file = f"{PROBES_DIR}/env_{probe_id}_{SCENARIO}.csv"

pd.DataFrame([env]).to_csv(
    probes_file,
    index=False
)